In [1]:
import os
import numpy as np
import pickle
import pandas as pd
from brainbox.io.one import SessionLoader
import gc
import concurrent.futures
import gzip

from segmentation_functions import merge_licks, resample_common_time, interpolate_nans, low_pass
from one.api import ONE
one = ONE(mode='remote')

In [2]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

paw_states = False

In [3]:
""" Load from Google Drive """
gdrive_path = "/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/"
filename = '1_bwm_qc_04-26-2026'
bwm_query = pickle.load(gzip.open(gdrive_path+filename, "rb"))

In [4]:
# Loop through animals
sessions = bwm_query['eid'].unique()

data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/'

# os.chdir(data_path)
files = os.listdir(data_path)
sessions_to_process = []

for s, sess in enumerate(sessions):
    file_path = one.eid2path(sess)

    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    filename = "design_matrix_" + str(sess) + '_'  + mouse_name
    if filename not in files:
        sessions_to_process.append((sess))

len(sessions_to_process)

324

In [67]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    """ LOAD VARIABLES """
    sl = SessionLoader(eid=session, one=one)
    sl.load_pose(views=['left', 'right'], tracker='lightningPose')
    sl.load_session_data(trials=True, wheel=True, motion_energy=True)

    # Check if all data is available
    if np.sum(sl.data_info['is_loaded']) >= 4:

        # Poses
        poses = sl.pose
        lc_t = np.asarray(poses['leftCamera']['times'])
        rc_t = np.asarray(poses['rightCamera']['times'])
        left_fr = int(1/np.nanmean(np.diff(lc_t)))
        right_fr = int(1/np.nanmean(np.diff(rc_t)))
        
        # To keep cutoff at 30 Hz, right camera should have more than 60 Hz
        if left_fr > 55 and right_fr > 60:

            # Motion energy
            me = sl.motion_energy
            mel_t = lc_t
            mer_t = rc_t
            if left_fr > 60:
                motion_energy_l = low_pass(interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr),
                                        cutoff=30, sf=left_fr)
            else:
                motion_energy_l = interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr)
            motion_energy_r = low_pass(interpolate_nans(me['rightCamera']['whiskerMotionEnergy'], right_fr), 
                                            cutoff=30, sf=right_fr)
            # Licks
            features = ['tongue_end_l_x', 'tongue_end_l_y', 'tongue_end_r_x', 'tongue_end_r_y']
            lick_t, licks = merge_licks(poses, features, common_fs=150)
            # Paws  
            if left_fr > 60:    
                l_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr), 
                                                cutoff=30, sf=left_fr)
                l_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr), 
                                                cutoff=30, sf=left_fr)
            else:
                l_paw_x = interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr)
                l_paw_y = interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr)
            r_paw_x = low_pass(interpolate_nans(poses['rightCamera']['paw_r_x'], right_fr), 
                                            cutoff=30, sf=right_fr)
            r_paw_y = low_pass(interpolate_nans(poses['rightCamera']['paw_r_y'], right_fr), 
                                            cutoff=30, sf=right_fr)
            l_paw_t = lc_t
            r_paw_t = rc_t
            # Wheel
            wheel = sl.wheel
            wheel_t = np.asarray(wheel['times'], dtype=np.float64)
            wheel_vel = wheel['velocity'].astype(np.float32)
            # Paw states
            if paw_states:
                times_l = one.load_dataset(session, '_ibl_%sCamera.times.npy' % 'left')
                times_r = one.load_dataset(session, '_ibl_%sCamera.times.npy' % 'right')
                assert np.all(lc_t==times_l)
                assert np.all(rc_t==times_r)
                paws_states_leftCamera = one.load_dataset(session, '_ibl_%sCamera.pawstates.pqt' % 'left')
                paws_states_rightCamera = one.load_dataset(session, '_ibl_%sCamera.pawstates.pqt' % 'right')
                paws_states_leftCamera['times'] = times_l
                paws_states_rightCamera['times'] = times_r

            # Common resampling window
            onset = max(lc_t.min(), rc_t.min(), wheel_t.min(), lick_t.min())
            offset = min(lc_t.max(), rc_t.max(), wheel_t.max(), lick_t.max())
            fs = 60
            ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

            # Restrict to time window
            def restrict(t, x):
                mask = (t >= onset) & (t <= offset)
                return t[mask], x[mask]

            mel_t, motion_energy_l = restrict(mel_t, motion_energy_l)
            mer_t, motion_energy_r = restrict(mer_t, motion_energy_r)
            wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)
            l_paw_t_x, l_paw_x = restrict(l_paw_t, l_paw_x)
            l_paw_t_y, l_paw_y = restrict(l_paw_t, l_paw_y)
            r_paw_t_x, r_paw_x = restrict(r_paw_t, r_paw_x)
            r_paw_t_y, r_paw_y = restrict(r_paw_t, r_paw_y)
            lick_t, licks = restrict(lick_t, licks)

            # Resample
            mel_down, rt = resample_common_time(ref_t, mel_t, motion_energy_l, kind='linear')
            mer_down, _ = resample_common_time(ref_t, mer_t, motion_energy_r, kind='linear')
            wh_down, _ = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')
            lk_down, _ = resample_common_time(ref_t, lick_t, licks, kind='nearest')
            lpx_down, _ = resample_common_time(ref_t, l_paw_t_x, l_paw_x, kind='linear')
            lpy_down, _ = resample_common_time(ref_t, l_paw_t_y, l_paw_y, kind='linear')
            rpx_down, _ = resample_common_time(ref_t, r_paw_t_x, r_paw_x, kind='linear')
            rpy_down, _ = resample_common_time(ref_t, r_paw_t_y, r_paw_y, kind='linear')

            # Create design matrix
            design_matrix = pd.DataFrame({
                'Bin': rt,
                'Lick count': lk_down.astype(np.int8),
                'avg_wheel_vel': wh_down,
                'whisker_me': mel_down,  # zscore(mel_down, nan_policy='omit'),
                'avg_whisker_me': np.nanmean([mel_down, mer_down], axis=0),  # zscore(np.nanmean([mel_down, mer_down], axis=0), nan_policy='omit'),
                'l_paw_x': lpx_down,
                'l_paw_y': lpy_down,
                'r_paw_x': rpx_down,
                'r_paw_y': rpy_down,
            })

            if paw_states:
                # Append paw states
                paw_vars = ['paw_r_still', 'paw_r_move', 'paw_r_wheel_turn', 'paw_r_groom',
                    'paw_r_still_ens_var', 'paw_r_move_ens_var', 'paw_r_wheel_turn_ens_var', 'paw_r_groom_ens_var',
                    'paw_l_still', 'paw_l_move', 'paw_l_wheel_turn', 'paw_l_groom',
                    'paw_l_still_ens_var', 'paw_l_move_ens_var', 'paw_l_wheel_turn_ens_var', 'paw_l_groom_ens_var']
                for s, state in enumerate(paw_vars):
                    resampled, _ = resample_common_time(ref_t, paws_states_leftCamera['times'], paws_states_leftCamera[state], kind='linear')
                    var_name = 'leftCam_'+state
                    design_matrix[var_name] = resampled
                    resampled, _ = resample_common_time(ref_t, paws_states_rightCamera['times'], paws_states_rightCamera[state], kind='linear')
                    var_name = 'rightCam_'+state
                    design_matrix[var_name] = resampled
        
            # """ LOAD TRIAL DATA """
            session_trials = sl.trials
            session_start = list(session_trials['goCueTrigger_times'])[0]
            design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

            """ SAVE DATA """       
            # Save unnormalized design matrix
            filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
            design_matrix.to_parquet(filename, compression='gzip')  

            # Save trials
            filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
            session_trials.to_parquet(filename, compression='gzip')  
            
            del design_matrix, session_trials, sl
            gc.collect()

    else:
        print('Data missing for session '+session)  


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [ ]:
for s, session in enumerate(sessions_to_process):
    process_design_matrix(session)